# CenterLoRA-ESM2 v1 — full-15 three-seed benchmark

This notebook trains seeds 42, 123, and 2026 for all 15 epochs and retains each seed's best development checkpoint. It uses only protein-grouped splits of the released training set. It neither reads the independent test nor performs a full-data refit. A GPU runtime is required.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Private-repository access uses a fine-grained read-only GitHub token stored in Colab Secrets as `GITHUB_TOKEN`, with notebook access enabled.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
print('Project:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json
import platform
import sys
import peft
import sklearn
import torch
import transformers

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('A GPU runtime is required for this benchmark.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Train all three seeds for 15 epochs

The stable Drive path and `--resume` make the suite restartable. Completed seeds are reused after a disconnection; an incomplete seed starts again from epoch 1. Seed 42 is intentionally retrained because its earlier screen stopped after epoch 10.

In [ ]:
BENCHMARK_ROOT = (
    Path('/content/drive/MyDrive/MMUbiPred-research/benchmarks')
    / 'center-lora-esm2-full15-three-seed-v1'
)
BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)

run_live([
    sys.executable, '-u', 'experiment/scripts/run_validation_benchmark.py',
    '--suite', 'experiment/configs/center_lora_esm2_full15_three_seed_v1.json',
    '--output-root', BENCHMARK_ROOT,
    '--resume',
])
print('Persistent benchmark root:', BENCHMARK_ROOT)

In [ ]:
CANDIDATE_SUMMARY_PATH = BENCHMARK_ROOT / 'summary.json'
candidate_summary = json.loads(CANDIDATE_SUMMARY_PATH.read_text())
display(candidate_summary['runs'])
display(candidate_summary['aggregates'])
display({
    'comparison_valid': candidate_summary['comparison_valid'],
    'collapsed_runs': candidate_summary['collapsed_runs'],
    'locked_test_accessed': False,
})

## Merge with the preserved stabilized baseline

The cell searches the existing Drive benchmark archive for a valid MMUbiPred-compatible three-seed summary. If automatic discovery fails, `REFERENCE_SUMMARY_PATH` accepts the preserved `mmubipred_stability_v2/summary.json` or `paired_baseline_stabilized_v2/summary.json` path.

In [ ]:
REFERENCE_SUMMARY_PATH = None  # Example: Path('/content/drive/MyDrive/.../summary.json')
BASELINE_MODEL = 'mmubipred_compatible'
EXPECTED_SEEDS = {42, 123, 2026}

def valid_baseline_summary(path):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        return False
    runs = [run for run in payload.get('runs', []) if run.get('model') == BASELINE_MODEL]
    seeds = {int(run['seed']) for run in runs}
    collapsed = any(
        bool(run.get('collapsed', False))
        or (
            float(run['fixed_sensitivity']) == 1.0
            and float(run['fixed_specificity']) == 0.0
        )
        or (
            float(run['fixed_sensitivity']) == 0.0
            and float(run['fixed_specificity']) == 1.0
        )
        for run in runs
    )
    return seeds == EXPECTED_SEEDS and not collapsed

if REFERENCE_SUMMARY_PATH is None:
    archive_root = Path('/content/drive/MyDrive/MMUbiPred-research/benchmarks')
    matches = [
        path for path in archive_root.rglob('summary.json')
        if path != CANDIDATE_SUMMARY_PATH and valid_baseline_summary(path)
    ]
    if not matches:
        raise FileNotFoundError(
            'No valid three-seed baseline summary was found. Set REFERENCE_SUMMARY_PATH manually.'
        )
    REFERENCE_SUMMARY_PATH = max(matches, key=lambda path: path.stat().st_mtime)
else:
    REFERENCE_SUMMARY_PATH = Path(REFERENCE_SUMMARY_PATH)
    if not valid_baseline_summary(REFERENCE_SUMMARY_PATH):
        raise ValueError('REFERENCE_SUMMARY_PATH is not a valid stable three-seed baseline.')

print('Using baseline summary:', REFERENCE_SUMMARY_PATH)
PAIRED_ROOT = BENCHMARK_ROOT / 'paired_with_stabilized_baseline'
run_live([
    sys.executable, '-u', 'experiment/scripts/merge_validation_benchmarks.py',
    '--reference-summary', REFERENCE_SUMMARY_PATH,
    '--reference-model', BASELINE_MODEL,
    '--candidate-summary', CANDIDATE_SUMMARY_PATH,
    '--candidate-model', 'center_lora_esm2_full15_v1',
    '--benchmark-name', 'center_lora_esm2_full15_vs_stabilized_baseline_v1',
    '--output-dir', PAIRED_ROOT,
])

In [ ]:
PAIRED_SUMMARY_PATH = PAIRED_ROOT / 'summary.json'
paired = json.loads(PAIRED_SUMMARY_PATH.read_text())
display(paired['runs'])
display(paired['aggregates'])
display(paired['paired_differences'])

by_model_seed = {
    (run['model'], int(run['seed'])): run for run in paired['runs']
}
seed_differences = {
    seed: (
        float(by_model_seed[('center_lora_esm2_full15_v1', seed)]['fixed_mcc'])
        - float(by_model_seed[(BASELINE_MODEL, seed)]['fixed_mcc'])
    )
    for seed in sorted(EXPECTED_SEEDS)
}
mean_difference = paired['paired_differences']['center_lora_esm2_full15_v1'][
    'candidate_minus_reference'
]['fixed_mcc']['mean']
positive_seeds = sum(value > 0 for value in seed_differences.values())
display({
    'comparison_valid': paired['comparison_valid'],
    'collapsed_runs': paired['collapsed_runs'],
    'fixed_mcc_candidate_minus_baseline_by_seed': seed_differences,
    'mean_fixed_mcc_difference': mean_difference,
    'positive_seed_count': positive_seeds,
    'locked_test_accessed': False,
})

if paired['comparison_valid'] and mean_difference > 0 and positive_seeds >= 2:
    print('DEVELOPMENT GATE PASSED: final refit or test access requires documented output review.')
else:
    print('DEVELOPMENT GATE NOT PASSED: independent-test evaluation is not authorized.')